# Sesión 4 — Streaming e inferencia en tiempo real (tutorial)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este notebook es la **parte teórico-práctica guiada** de la Sesión 4. Construimos un pipeline de *scoring* en **streaming**: un **productor** emite eventos sin fin, un **consumidor** puntúa el modelo sobre cada uno (**inferencia online**) y **agregamos métricas por ventana** (throughput, latencia p95 y un indicador simple de *drift*). Primero **en memoria** (corre en cualquier lado) y después con **Kafka/Redpanda en Docker** (el camino de producción).

> **Entorno (uv):**
> ```bash
> uv add scikit-learn joblib numpy kafka-python
> ```
> Ejecuta con el kernel de uv (ver *Puesta en marcha* del README raíz). La siguiente celda instala lo necesario si corres el notebook tal cual.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn", "joblib", "numpy"])
print("Dependencias base listas. (kafka-python solo para la parte con Docker.)")

## 1. Batch vs streaming: el cambio de mentalidad

Hasta ahora servimos el modelo bajo un patrón **request→response** (REST, GraphQL, gRPC): alguien pregunta, el modelo responde. Pero muchos datos llegan como un **flujo continuo** de eventos —transacciones, clics, sensores— **sin principio ni fin**.

- **Batch (datos en reposo):** se almacena todo y se procesa a intervalos. Alta latencia; el dato envejece.
- **Streaming (datos en movimiento):** la *consulta* está siempre corriendo y el dato pasa por ella **una vez**, al llegar. Baja latencia; el resultado siempre está fresco.

Para fraude, alertas o recomendación en vivo, el **valor del dato decae** rápido tras el evento: por eso el tiempo real.

## 2. Un modelo para puntuar

Usamos un modelo mínimo (en tu Mini-TP será el tuyo). Lo cargamos **una sola vez**: puntuará millones de eventos.

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np, joblib

rng = np.random.RandomState(0)
X = rng.randn(300, 3); y = (X.sum(axis=1) > 0).astype(int)
modelo = LogisticRegression().fit(X, y)
joblib.dump(modelo, "model.pkl")
print("Modelo listo (3 features).")

## 3. El flujo: productor y consumidor (en memoria)

Simulamos el flujo con una **cola** (`queue.Queue`) y dos hilos: el **productor** emite eventos y el **consumidor** los toma a medida que llegan. Cada evento trae sus *features* y una **marca de tiempo de origen** (*event time*).

Para que el notebook termine, el productor emite un número fijo de eventos y luego una **señal de fin** (`STOP`); en producción el bucle no termina. A mitad del flujo introducimos un **desplazamiento en la media** de las entradas, para luego **detectarlo como drift**.

In [ ]:
import queue, threading, time, random
from collections import deque

stream = queue.Queue()
STOP = object()
N = 800  # eventos a emitir (en producción: sin fin)

def productor():
    for i in range(N):
        mu = 0.0 if i < N // 2 else 1.2      # drift a mitad de camino
        evento = {"values": [random.gauss(mu, 1) for _ in range(3)],
                  "t": time.time()}
        stream.put(evento)
        time.sleep(0.0005)
    stream.put(STOP)
print("Productor definido.")

## 4. Inferencia online + ventana deslizante

El consumidor puntúa el modelo sobre **cada** evento (inferencia online) y mantiene una **ventana deslizante** de 1 segundo con lo reciente. Sobre esa ventana calculamos, de forma continua:

- **throughput**: eventos por segundo,
- **latencia p95**: cuánto tarda el scoring,
- **drift** simple: media de la primera *feature* en la ventana (si se dispara, algo cambió en la entrada).

In [ ]:
import numpy as np

latencias = []
ventana = deque()          # (event_time, feature0, pred)
snapshots = []             # métricas por corte

def consumidor():
    procesados = 0
    while True:
        ev = stream.get()
        if ev is STOP:
            break
        t0 = time.perf_counter()
        pred = modelo.predict([ev["values"]])[0]        # inferencia online
        latencias.append((time.perf_counter() - t0) * 1000)

        ventana.append((ev["t"], ev["values"][0], pred))
        ahora = ev["t"]
        while ventana and ahora - ventana[0][0] > 1.0:   # ventana deslizante 1s
            ventana.popleft()

        procesados += 1
        if procesados % 200 == 0:                        # un corte cada 200 eventos
            media_f0 = np.mean([w[1] for w in ventana])
            snapshots.append((procesados, len(ventana), media_f0))

print("Consumidor definido.")

In [ ]:
t0 = time.time()
pt = threading.Thread(target=productor)
ct = threading.Thread(target=consumidor)
pt.start(); ct.start(); pt.join(); ct.join()
dur = time.time() - t0

print(f"Eventos procesados : {len(latencias)}")
print(f"Duración           : {dur:.2f} s")
print(f"Throughput         : {len(latencias)/dur:.0f} ev/s")
print(f"Latencia p95       : {np.percentile(latencias, 95):.3f} ms")
print()
print("Drift (media de feature0 por ventana, debería subir tras la mitad):")
for proc, n, media in snapshots:
    marca = "  <-- drift" if media > 0.6 else ""
    print(f"  evento {proc:4d} | ventana={n:3d} | media_f0={media:+.2f}{marca}")

> Observa cómo la **media de la primera feature** salta cuando el productor cambia la distribución: así se ve, en vivo, un **drift de entradas**. En producción compararías esta media (u otra estadística) contra la del entrenamiento y **alertarías** al cruzar un umbral.

## 5. Tiempo, ventanas y estado (los conceptos)

- **Event time vs processing time:** agregamos por *cuándo ocurrió* el evento (event time), no por cuándo llegó. Un **watermark** declara "ya llegó todo lo anterior a t" y permite **cerrar ventanas** tolerando eventos tardíos/desordenados.
- **Tumbling vs sliding:** ventanas fijas que no se solapan (reportes por bloque) vs deslizantes que se solapan (métricas móviles, como la de arriba).
- **Stateless vs stateful:** filtrar/transformar un evento no guarda estado (fácil de paralelizar); **agregar por ventana** o **detectar patrones** sí lo guarda (el operador recuerda lo recibido). Todo operador con estado debe **expirar lo viejo**, o la memoria crece sin límite.

## 6. El camino de producción: Kafka (Redpanda) en Docker

En memoria alcanza para aprender, pero en producción el flujo pasa por un **broker**: guarda los eventos en un **log particionado** (topic), desacopla productores de consumidores y permite **escalar** y **reprocesar**. Usamos **Redpanda**, que habla la **API de Kafka** y es más liviano.

```bash
# 1) levantar el broker (en una terminal aparte)
docker run -d --name redpanda -p 9092:9092 \
  redpandadata/redpanda redpanda start --overprovisioned --smp 1 --check=false

# 2) dependencias
uv add kafka-python
```

La celda siguiente **solo corre si hay un broker en `localhost:9092`**; si no, la salta sin romper el notebook.

In [ ]:
import socket
def hay_broker(host="localhost", port=9092):
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

if not hay_broker():
    print("No hay broker en localhost:9092 -> se omite la parte de Kafka.")
    print("Levanta Redpanda con el comando docker de arriba y vuelve a correr esta celda.")
else:
    from kafka import KafkaProducer, KafkaConsumer
    import json, threading, time

    prod = KafkaProducer(bootstrap_servers="localhost:9092",
                         value_serializer=lambda v: json.dumps(v).encode())
    for i in range(50):
        prod.send("eventos", {"values": [float(x) for x in np.random.randn(3)]})
    prod.flush()

    cons = KafkaConsumer("eventos", bootstrap_servers="localhost:9092",
                         auto_offset_reset="earliest",
                         value_deserializer=lambda b: json.loads(b.decode()),
                         consumer_timeout_ms=3000)
    n = 0
    for msg in cons:
        pred = modelo.predict([msg.value["values"]])[0]   # mismo scoring
        n += 1
    print(f"Consumidos y puntuados {n} mensajes del topic 'eventos'.")
    print("Mismo código de scoring; solo cambió de dónde llegan los eventos.")

## 7. Errores comunes

- **Confundir event time con processing time** al agregar.
- **Ventanas sin watermark:** nunca cierran o pierden eventos tardíos.
- **Estado que crece sin límite:** todo operador stateful debe **expirar** lo viejo.
- **Consumidor no idempotente:** Kafka entrega *at-least-once*; reprocesar no debe duplicar efectos.
- **Ignorar el backpressure:** si el consumidor no da abasto, hay que frenar o escalar (más particiones/consumidores).
- **Cargar el modelo por evento:** cárgalo una vez.

---

### Del tutorial al Mini-TP 4

Este pipeline es el molde: en el **Mini-TP 4** consumes un flujo con las features de **tu** modelo, lo puntúas online, agregas métricas por ventana, disparas una alerta y comparas contra batch. Starter en **`mini_tp4_actividad.ipynb`**. Y hoy arranca el **TP integrador (Hito #1)**: equipo + arquitectura de referencia.